In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import json
import os
import re
import sys
import time
import unicodedata
from getpass import getpass
from collections import Counter

import pandas as pd
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from rapidfuzz import fuzz, process

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()

WIKIPEDIA_DIR = PROJECT_ROOT / "3 Wikipedia"
LDA_DIR = PROJECT_ROOT / "4 LDA Topic Analysis"
LLM_WIKIPEDIA_DIR = PROJECT_ROOT / "5 LLM Wikipedia analysis"
OUTPUTS_DIR = LLM_WIKIPEDIA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
for folder in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    WIKIPEDIA_DIR / "outputs" / "tables" / "mep_wikipedia_final.xlsx",
    WIKIPEDIA_DIR / "mep_wikipedia_final.xlsx",
]

POLARISATION_CANDIDATES = [
    LDA_DIR / "outputs" / "tables" / "polarisation_scores_scored_only.csv",
    LDA_DIR / "polarisation_scores_scored_only.csv",
]

INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])
POLARISATION_SCORED_ONLY_PATH = next((p for p in POLARISATION_CANDIDATES if p.exists()), POLARISATION_CANDIDATES[0])

OUTPUT_PATH = TABLES_DIR / "wikipedia_extracted.xlsx"
PARTIAL_PATH = TABLES_DIR / "wikipedia_extracted_partial.xlsx"
FINAL_POLARISATION_PATH = TABLES_DIR / "extracted_polarisation_final.xlsx"
FINAL_POLARISATION_MANUAL_PATH = LLM_WIKIPEDIA_DIR / "extracted_polarisation_final_manually_improved_1906.xlsx"
AUDIT_OUTPUT_PATH = TABLES_DIR / "polarisation_vs_extracted_match_audit.csv"
LLM_AUDIT_JSONL_PATH = TABLES_DIR / "llm_extraction_audit.jsonl"

assert INPUT_PATH.exists(), f"Wikipedia input missing: {INPUT_PATH}"
assert POLARISATION_SCORED_ONLY_PATH.exists(), (
    f"Polarisation input missing: {POLARISATION_SCORED_ONLY_PATH}"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file:   {INPUT_PATH}")
print(f"Output file:  {OUTPUT_PATH}")

Project root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics
Input file:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\mep_wikipedia_final.xlsx
Output file:  C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\wikipedia_extracted.xlsx


# Wikipedia Biographical Extraction (LLM)

This notebook extracts structured biographical information from Wikipedia pages and creates a final table for scored MEPs.

Run from top to bottom. Paths are configured once in the next cell for portability and easier GitHub reuse.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [31]:
# Always prompt for a fresh key in this kernel run (visible input).
entered = input("Enter OPENROUTER_API_KEY (visible): ").strip()
if not entered:
    raise RuntimeError("No key entered. Please provide OPENROUTER_API_KEY.")

# Overwrite both env vars so downstream cells use the same value.
os.environ["OPENROUTER_API_KEY"] = entered
os.environ["OPENROUTER_KEY"] = entered

key_now = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
print("OPENROUTER_API_KEY set in this kernel:", bool(key_now))
print("OPENROUTER_API_KEY length:", len(key_now))

# Quick auth sanity check (does not print the key).
try:
    _resp = requests.get(
        "https://openrouter.ai/api/v1/models",
        headers={"Authorization": f"Bearer {key_now}"},
        timeout=20,
    )
    print("OpenRouter auth check status:", _resp.status_code)
    if _resp.status_code != 200:
        print("OpenRouter error preview:", (_resp.text or "")[:240])
except Exception as e:
    print("OpenRouter auth check failed:", str(e))

OPENROUTER_API_KEY set in this kernel: True
OPENROUTER_API_KEY length: 73
OpenRouter auth check status: 200


In [46]:
# ------------------------------------------------------------
# Free-model smoke test (run immediately after the API key cell)
# ------------------------------------------------------------

def test_openrouter_free_model_smoke() -> dict:
    result = {
        "key_present": False,
        "key_length": 0,
        "models_status": None,
        "free_models_found": [],
        "chosen_model": None,
        "attempted_models": [],
        "chat_status": None,
        "chat_ok": False,
        "chat_preview": None,
        "error": None,
    }

    key = (os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENROUTER_KEY") or "").strip()
    result["key_present"] = bool(key)
    result["key_length"] = len(key)

    if not key:
        result["error"] = "No OPENROUTER_API_KEY/OPENROUTER_KEY found in this kernel."
        return result

    try:
        models_resp = requests.get(
            "https://openrouter.ai/api/v1/models",
            headers={"Authorization": f"Bearer {key}"},
            timeout=20,
        )
        result["models_status"] = models_resp.status_code
        if models_resp.status_code != 200:
            result["error"] = (models_resp.text or "")[:300]
            return result

        data = models_resp.json().get("data", [])
        free_models = [
            m.get("id", "") for m in data
            if isinstance(m, dict) and str(m.get("id", "")).endswith(":free")
        ]
        result["free_models_found"] = free_models[:10]

        if not free_models:
            result["error"] = "No free models were returned by /models."
            return result

        for chosen_model in free_models[:8]:
            result["attempted_models"].append(chosen_model)
            chat_payload = {
                "model": chosen_model,
                "messages": [
                    {"role": "system", "content": "Reply with exactly OK."},
                    {"role": "user", "content": "Ping"},
                ],
                "temperature": 0.0,
                "max_tokens": 12,
            }
            chat_resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {key}",
                    "Content-Type": "application/json",
                },
                json=chat_payload,
                timeout=30,
            )
            result["chat_status"] = chat_resp.status_code
            result["chosen_model"] = chosen_model

            if chat_resp.status_code != 200:
                continue

            data = chat_resp.json()
            content = (((data.get("choices") or [{}])[0].get("message") or {}).get("content") or "").strip()
            if content:
                result["chat_preview"] = content[:120]
                result["chat_ok"] = True
                break

        if not result["chat_ok"]:
            if result["chat_status"] == 200:
                result["error"] = "Free models returned 200 but no content from the first several attempts."
            else:
                result["error"] = "No free model returned a valid chat response."
    except Exception as e:
        result["error"] = str(e)

    return result

preflight = test_openrouter_free_model_smoke()
print("OpenRouter free-model smoke test:")
for k, v in preflight.items():
    print(f"- {k}: {v}")

if not preflight["chat_ok"]:
    raise RuntimeError("Free-model smoke test failed: fix key/auth/model access before running extraction cells.")

OpenRouter free-model smoke test:
- key_present: True
- key_length: 73
- models_status: 200
- free_models_found: ['cohere/north-mini-code:free', 'nex-agi/nex-n2-pro:free', 'nvidia/nemotron-3.5-content-safety:free', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'poolside/laguna-xs.2:free', 'poolside/laguna-m.1:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'nvidia/nemotron-3-super-120b-a12b:free']
- chosen_model: nex-agi/nex-n2-pro:free
- attempted_models: ['cohere/north-mini-code:free', 'nex-agi/nex-n2-pro:free']
- chat_status: 200
- chat_ok: True
- chat_preview: OK
- error: None


In [47]:

# -------------------
# Main extraction configuration and shared helpers
# -------------------
N_FIRST = 1919  # limit rows for test runs if needed

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "meta-llama/llama-4-maverick:free"

# Do not hardcode secrets in notebooks.
API_KEY = (
    (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    or (os.environ.get("OPENROUTER_KEY") or "").strip()
 )

client = None
if not API_KEY:
    print("OPENROUTER_API_KEY not found. Set it before running extraction cells.")
else:
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

SYSTEM = (
    "You extract and classify structured biographical info from Wikipedia text. "
    "Return only valid JSON that matches the requested schema exactly. "
    "Use ONLY the allowed bucket values. "
    "If unknown / not stated in the text, use null (or empty list where applicable). "
    "Do not invent details. "
    "For classifications, choose the closest bucket only when the text supports it. "
    "Always provide short evidence snippets copied from the text (max ~200 chars each)."
 )

ALLOWED = {
  "gender_bucket": ["Male", "Female", "Other or Not stated"],
  "region_bucket": ["Northern Europe", "Western Europe", "Southern Europe", "Central & Eastern Europe", "Baltic"],
  "education_level_bucket": ["No higher education listed", "Bachelor", "Master", "Doctorate"],
  "education_field_bucket": [
      "Economics or Business", "Law", "Political Science or Public Administration",
      "Sociology or Demography or Social Sciences", "Humanities", "STEM", "Medicine or Health",
      "Education or Teaching", "Other or Unclear",
  ],
  "professional_background_bucket": [
      "Business or Finance", "Law", "Academia or Research", "Public Administration", "NGO or Civil Society",
      "Trade Union", "Journalism or Media", "Education Sector", "Health Sector", "Manual or Technical",
      "Career Politician", "Other or Unclear",
  ],
  "religion_bucket": ["Christian", "Other religion", "Explicitly secular", "Not mentioned"],
  "parental_status_bucket": ["Has children", "No children", "Not mentioned"],
  "migration_background_bucket": ["Born abroad", "Parent foreign-born", "None mentioned"],
  "international_experience_bucket": ["Studied abroad", "Worked abroad", "EU institutional career", "None significant mentioned"],
  "military_background_bucket": ["Yes", "No", "Not mentioned"],
  "prior_political_level_bucket": ["Local only", "National parliament", "Government minister", "No prior elected office", "Unclear"],
  "demography_expertise_bucket": ["Demography specialist", "Social policy background", "None"],
}

def html_to_text(html: str) -> str:
    if not isinstance(html, str) or not html.strip():
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def safe_json_loads(s: str) -> dict:
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if match:
        s = match.group(0).strip()

    try:
        return json.loads(s)
    except json.JSONDecodeError:
        s2 = re.sub(r",\s*([}\]])", r"\1", s)
        s2 = s2.replace("\u00a0", " ").strip()
        return json.loads(s2)

def is_mep_from_text(text: str) -> bool:
    if not text:
        return False
    patterns = [
        r"\bMember of the European Parliament\b",
        r"\bMEP\b",
        r"\bMembers of the European Parliament\b",
        r"\bEuropean Parliament\b.*\bmember\b",
        r"\b(member|deputy)\b.*\bEuropean Parliament\b",
    ]
    text_flat = " ".join(text.split())
    return any(re.search(p, text_flat, flags=re.IGNORECASE) for p in patterns)

def flatten_extract(d: dict) -> dict:
    out = {}
    out["full_name_extracted"] = d.get("full_name")
    out["birth_date_extracted"] = d.get("birth_date")
    out["place_of_birth_extracted"] = d.get("place_of_birth")
    out["country_of_origin_extracted"] = d.get("country_of_origin")
    out["ep_group_extracted"] = d.get("ep_group")
    out["mep_since_extracted"] = d.get("mep_since")
    out["occupations_extracted"] = json.dumps(d.get("occupations", []), ensure_ascii=False)
    out["parties_extracted"] = json.dumps(d.get("parties", []), ensure_ascii=False)
    out["education_extracted"] = json.dumps(d.get("education", []), ensure_ascii=False)
    out["confidence_extracted"] = d.get("confidence")

    buckets = d.get("buckets", {}) or {}
    for key, value in buckets.items():
        out[key] = value

    evidence = d.get("evidence", {}) or {}
    for key, value in evidence.items():
        out[f"evidence_{key}"] = value

    return out


In [4]:
from datetime import datetime, timezone

PROMPT_VERSION = "v1.0.0"
REQUIRED_EXTRACT_FIELDS = [
    "full_name",
    "birth_date",
    "country_of_origin",
    "buckets",
]


def extraction_quality_flags(parsed: dict) -> dict:
    if not isinstance(parsed, dict):
        return {
            "malformed_output": True,
            "missing_required_fields": REQUIRED_EXTRACT_FIELDS,
        }

    missing = [k for k in REQUIRED_EXTRACT_FIELDS if k not in parsed or parsed.get(k) in [None, "", []]]
    return {
        "malformed_output": False,
        "missing_required_fields": missing,
    }


def append_audit_jsonl(entry: dict) -> None:
    LLM_AUDIT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(LLM_AUDIT_JSONL_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Prompt version: {PROMPT_VERSION}")
print(f"Audit log path: {LLM_AUDIT_JSONL_PATH}")

Prompt version: v1.0.0
Audit log path: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\llm_extraction_audit.jsonl


In [17]:

# Prefer known working free models first; dynamic discovery appends more.
STATIC_MODEL_CANDIDATES = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "openai/gpt-oss-20b:free",
    "qwen/qwen3-coder:free",
    "google/gemma-4-31b-it:free",
    "meta-llama/llama-3.2-3b-instruct:free",
]

# Keep requests small enough for free-tier limits.
MAX_INPUT_CHARS = 18000
MAX_OUTPUT_TOKENS = 1200
MAX_RETRIES_PER_MODEL = 2
BACKOFF_SECONDS = 1.5
LIVE_FREE_LIMIT = 30

LAST_MODEL_USED = None

def _discover_live_free_models(limit: int = LIVE_FREE_LIMIT) -> list[str]:
    """Fetch currently live free model slugs from OpenRouter."""
    if not API_KEY:
        return []

    try:
        resp = requests.get(
            f"{BASE_URL}/models",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
        ids = [m.get("id", "") for m in data if isinstance(m, dict)]
        ids = [m for m in ids if m.endswith(":free")]
        return ids[:limit]
    except Exception:
        return []

def _build_model_candidates() -> list[str]:
    live = _discover_live_free_models()
    merged = []
    for m in STATIC_MODEL_CANDIDATES + live:
        m = (m or "").strip()
        if m and m not in merged:
            merged.append(m)
    return merged

MODEL_CANDIDATES = _build_model_candidates()

def _is_non_retriable_model_error(err_text: str) -> bool:
    t = err_text.lower()
    return (
        "error code: 404" in t
        or "no endpoints found" in t
        or "unavailable for free" in t
        or "model not found" in t
    )

def _build_extraction_prompt(raw_text: str, name_hint: str, lang: str) -> str:
    return f"""
Name hint: {name_hint}
Language of page: {lang}

Task:
Return ONLY valid JSON matching the schema. No commentary.

Allowed buckets (exact strings):
{json.dumps(ALLOWED, ensure_ascii=False, indent=2)}

JSON schema:
{{
  "full_name": string|null,
  "birth_date": string|null,
  "place_of_birth": string|null,
  "country_of_origin": string|null,
  "education": [{{"degree": string|null,"field": string|null,"institution": string|null,"country": string|null,"start_year": integer|null,"end_year": integer|null,"notes": string|null}}],
  "occupations": [string],
  "parties": [string],
  "ep_group": string|null,
  "mep_since": string|null,
  "buckets": {{
    "gender_bucket": string|null,
    "region_bucket": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "religion_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "military_background_bucket": string|null,
    "prior_political_level_bucket": string|null,
    "demography_expertise_bucket": string|null
  }},
  "confidence": number,
  "evidence": {{
    "birth_date": string|null,
    "country_of_origin": string|null,
    "ep_group": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "prior_political_level_bucket": string|null
  }}
}}

Wikipedia text:
\"\"\"{raw_text[:MAX_INPUT_CHARS]}\"\"\"
""".strip()

def _call_model(model_name: str, prompt: str):
    if client is None:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set in this kernel. "
            "Set it and rerun from the configuration cell."
        )

    return client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=MAX_OUTPUT_TOKENS,
    )

def extract_with_llm(raw_text: str, name_hint: str, lang: str) -> dict:
    global LAST_MODEL_USED

    if client is None:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set in this kernel. "
            "Set it and rerun from the configuration cell."
        )

    prompt = _build_extraction_prompt(raw_text, name_hint, lang)
    errors = []

    if not MODEL_CANDIDATES:
        raise RuntimeError("No free model candidates available right now.")

    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                resp = _call_model(model_name, prompt)
                content = (resp.choices[0].message.content or "").strip()
                if not content:
                    raise ValueError("Empty model output")

                try:
                    parsed = safe_json_loads(content)
                    LAST_MODEL_USED = model_name
                    return parsed
                except Exception:
                    fixer = f"""
Fix the following so it is strictly valid JSON.
Do not change meaning or values.
Return JSON only.

INPUT:
{content}
""".strip()

                    resp2 = client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {"role": "system", "content": "Return only valid JSON."},
                            {"role": "user", "content": fixer},
                        ],
                        temperature=0.0,
                        max_tokens=MAX_OUTPUT_TOKENS,
                    )
                    fixed = (resp2.choices[0].message.content or "").strip()
                    parsed = safe_json_loads(fixed)
                    LAST_MODEL_USED = model_name
                    return parsed

            except Exception as e:
                msg = str(e)
                if "OPENROUTER_API_KEY is not set" in msg:
                    raise RuntimeError(msg)

                errors.append(f"{model_name} attempt {attempt}: {msg}")
                if _is_non_retriable_model_error(msg):
                    break
                time.sleep(BACKOFF_SECONDS * attempt)

    short_errors = "\n".join(errors[-8:])
    raise RuntimeError(
        "All free model attempts failed. Last errors:\n" + short_errors
    )

In [19]:
# ------------------------------------------------------------
# Transport fallback: call OpenRouter directly via requests
# (keeps model selection/retry logic, avoids SDK auth-header issue)
# ------------------------------------------------------------

def _chat_content_via_requests(model_name: str, system_text: str, user_text: str) -> str:
    if not API_KEY:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set in this kernel. "
            "Set it and rerun from the configuration cell."
        )

    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        "temperature": 0.0,
        "max_tokens": MAX_OUTPUT_TOKENS,
    }

    resp = requests.post(
        f"{BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=90,
    )
    if resp.status_code >= 400:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:400]}")

    data = resp.json()
    choices = data.get("choices") or []
    if not choices:
        raise RuntimeError("No choices returned by OpenRouter")

    content = (((choices[0] or {}).get("message") or {}).get("content") or "").strip()
    if not content:
        raise RuntimeError("Empty model output")
    return content


def extract_with_llm(raw_text: str, name_hint: str, lang: str) -> dict:
    global LAST_MODEL_USED

    prompt = _build_extraction_prompt(raw_text, name_hint, lang)
    errors = []

    if not MODEL_CANDIDATES:
        raise RuntimeError("No free model candidates available right now.")

    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                content = _chat_content_via_requests(model_name, SYSTEM, prompt)

                try:
                    parsed = safe_json_loads(content)
                    LAST_MODEL_USED = model_name
                    return parsed
                except Exception:
                    fixer = f"""
Fix the following so it is strictly valid JSON.
Do not change meaning or values.
Return JSON only.

INPUT:
{content}
""".strip()

                    fixed = _chat_content_via_requests(
                        model_name,
                        "Return only valid JSON.",
                        fixer,
                    )
                    parsed = safe_json_loads(fixed)
                    LAST_MODEL_USED = model_name
                    return parsed

            except Exception as e:
                msg = str(e)
                errors.append(f"{model_name} attempt {attempt}: {msg}")
                if _is_non_retriable_model_error(msg):
                    break
                time.sleep(BACKOFF_SECONDS * attempt)

    short_errors = "\n".join(errors[-8:])
    raise RuntimeError(
        "All free model attempts failed. Last errors:\n" + short_errors
    )

In [ ]:
if client is None:
    raise RuntimeError(
        "OPENROUTER_API_KEY is not set in this kernel. "
        "Set it and rerun this extraction cell."
    )

df = pd.read_excel(INPUT_PATH)
df_first = df.head(N_FIRST).copy()

rows = []
audit_rows = []
for idx, row in df_first.iterrows():
    name_hint = str(row.get("name", "") or "")
    lang = str(row.get("wikipedia_language", "") or "")

    html_en = row.get("wikipedia_content_html_en")
    html_local = row.get("wikipedia_content_html")

    text_en = html_to_text(html_en) if isinstance(html_en, str) else ""
    text_local = html_to_text(html_local) if isinstance(html_local, str) else ""
    text = text_en + "\n\n" + text_local

    ts = datetime.now(timezone.utc).isoformat()
    status = "ok"
    err_msg = None
    extracted = None

    try:
        extracted = extract_with_llm(text, name_hint=name_hint, lang=lang)
        flat = flatten_extract(extracted)
        mep_flag = bool(extracted.get("mep_since") or extracted.get("ep_group")) or is_mep_from_text(text)
        flat["MEP"] = bool(mep_flag)
        quality = extraction_quality_flags(extracted)
        flat["needs_manual_review"] = bool(quality["malformed_output"] or len(quality["missing_required_fields"]) > 0)
        flat["missing_required_fields"] = json.dumps(quality["missing_required_fields"], ensure_ascii=False)
    except Exception as e:
        status = "error"
        err_msg = str(e)
        flat = {
            "MEP": pd.NA,
            "extract_error": err_msg,
            "needs_manual_review": True,
            "missing_required_fields": json.dumps(REQUIRED_EXTRACT_FIELDS, ensure_ascii=False),
        }

    rows.append(flat)

    audit_entry = {
        "timestamp_utc": ts,
        "name": name_hint,
        "wikipedia_language": lang,
        "model_used": globals().get("LAST_MODEL_USED", MODEL),
        "prompt_version": PROMPT_VERSION,
        "error_status": status,
        "error_message": err_msg,
        "raw_response": None if extracted is None else json.dumps(extracted, ensure_ascii=False),
        "parsed_output": None if extracted is None else extracted,
        "missing_required_fields": [] if extracted is None else extraction_quality_flags(extracted)["missing_required_fields"],
        "needs_manual_review": bool(status != "ok" or (extracted is not None and len(extraction_quality_flags(extracted)["missing_required_fields"]) > 0)),
    }
    append_audit_jsonl(audit_entry)
    audit_rows.append(audit_entry)

df_out = pd.concat([df_first.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
df_out.to_excel(OUTPUT_PATH, index=False)

assert len(df_out) > 0, "Extraction output is empty"
assert OUTPUT_PATH.exists(), f"Missing output file: {OUTPUT_PATH}"

print(f"Wrote: {OUTPUT_PATH}")
print(f"Audit rows written: {len(audit_rows)} -> {LLM_AUDIT_JSONL_PATH}")

KeyboardInterrupt: 

In [ ]:

partial_out = pd.concat([df_first.iloc[:len(rows)].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
partial_out.to_excel(PARTIAL_PATH, index=False)
print(f"Wrote partial: {PARTIAL_PATH} ({len(rows)} rows)")


Wrote partial: C:\Users\Gebruiker\OneDrive - UniversitÃƒÆ’Ã‚Â  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\5 LLM Wikipedia analysis\wikipedia_extracted_partial.xlsx (554 rows)


## Incremental extraction for polarisation-scored MEPs

This section builds extracted_polarisation_final.xlsx by:
1. Loading polarisation_scores_scored_only.csv.
2. Reusing already-processed MEPs from the partial extraction file.
3. Running extraction only for missing scored MEPs.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [33]:
# ------------------------------------------------------------
# Preflight check: do we need new LLM extraction now?
# ------------------------------------------------------------

def _norm_name_preflight(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def _match_key_preflight(x: str) -> str:
    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}
    tokens = [t for t in _norm_name_preflight(x).split() if t and t not in stop_tokens]
    return " ".join(sorted(tokens))


pol_df_pre = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()
pol_name_candidates_pre = ["mep_name_raw", "mep_name_clean", "name"]
pol_name_col_pre = next((c for c in pol_name_candidates_pre if c in pol_df_pre.columns), None)
if pol_name_col_pre is None:
    raise ValueError(f"Could not find a name column in polarisation file. Columns: {list(pol_df_pre.columns)}")

pol_df_pre["_name_key"] = pol_df_pre[pol_name_col_pre].fillna("").astype(str).map(_match_key_preflight)
pol_df_pre = pol_df_pre[pol_df_pre["_name_key"] != ""].copy()
pol_df_pre = pol_df_pre.drop_duplicates(subset=["_name_key"], keep="first")
target_keys_pre = set(pol_df_pre["_name_key"].tolist())

reuse_candidates = [
    PARTIAL_PATH,
    FINAL_POLARISATION_PATH,
    FINAL_POLARISATION_MANUAL_PATH,
]

reuse_frames = []
reuse_paths_used = []
for candidate_path in reuse_candidates:
    if candidate_path.exists():
        try:
            tmp_df = pd.read_excel(candidate_path)
            if not tmp_df.empty:
                reuse_frames.append(tmp_df)
                reuse_paths_used.append(str(candidate_path))
        except Exception as e:
            print(f"Could not read reuse candidate {candidate_path} ({e}).")

if reuse_frames:
    reuse_df_pre = pd.concat(reuse_frames, ignore_index=True)
else:
    reuse_df_pre = pd.DataFrame()

if "name" in reuse_df_pre.columns:
    reuse_name_col_pre = "name"
elif "full_name_extracted" in reuse_df_pre.columns:
    reuse_name_col_pre = "full_name_extracted"
elif "mep_name_raw" in reuse_df_pre.columns:
    reuse_name_col_pre = "mep_name_raw"
elif "mep_name_clean" in reuse_df_pre.columns:
    reuse_name_col_pre = "mep_name_clean"
else:
    reuse_name_col_pre = None

if reuse_name_col_pre is not None and not reuse_df_pre.empty:
    reuse_df_pre["_name_key"] = reuse_df_pre[reuse_name_col_pre].fillna("").astype(str).map(_match_key_preflight)
    processed_keys_pre = set(reuse_df_pre["_name_key"].dropna().astype(str).tolist())
else:
    processed_keys_pre = set()

missing_keys_pre = [k for k in target_keys_pre if k not in processed_keys_pre]
LLM_NEEDED_NOW = len(missing_keys_pre) > 0

print(f"Reuse files considered: {len(reuse_paths_used)}")
if reuse_paths_used:
    for p in reuse_paths_used:
        print(f"  - {p}")

print(f"Scored MEP targets: {len(target_keys_pre)}")
print(f"Already available from existing extracted files: {len(target_keys_pre) - len(missing_keys_pre)}")
print(f"Need new extraction now: {len(missing_keys_pre)}")
print(f"LLM needed now: {LLM_NEEDED_NOW}")

if missing_keys_pre:
    missing_name_df_pre = (
        pol_df_pre.loc[pol_df_pre["_name_key"].isin(missing_keys_pre), [pol_name_col_pre, "_name_key"]]
        .rename(columns={pol_name_col_pre: "mep_name"})
        .drop_duplicates(subset=["_name_key"], keep="first")
        .sort_values("mep_name")
        .reset_index(drop=True)
    )
    print("Missing scored MEP names:")
    display(missing_name_df_pre)

if not LLM_NEEDED_NOW:
    print("Recommendation: run incremental block with RUN_EXTRACTION = False.")
else:
    print("Recommendation: keep RUN_EXTRACTION = True and provide OPENROUTER_API_KEY only for missing names.")

Reuse files considered: 1
  - C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\extracted_polarisation_final.xlsx
Scored MEP targets: 158
Already available from existing extracted files: 155
Need new extraction now: 3
LLM needed now: True
Missing scored MEP names:


,mep_name,_name_key
0,Arturs Krišjānis Kariņš,arturs karins krisjanis
1,László Andor,andor laszlo
2,Pierfrancesco Majorino,majorino pierfrancesco


Recommendation: keep RUN_EXTRACTION = True and provide OPENROUTER_API_KEY only for missing names.


In [59]:
try:
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False

FUZZY_TOKEN_SET_THRESHOLD = 72
FUZZY_TOKEN_SORT_THRESHOLD = 70
FUZZY_PARTIAL_THRESHOLD = 82

# Default behavior: if preflight cell was run, use its recommendation.
# You can still override manually by setting RUN_EXTRACTION before this cell.
RUN_EXTRACTION = bool(globals().get("LLM_NEEDED_NOW", True))
CHECKPOINT_EVERY = 1

def _norm_name(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def _match_key(x: str) -> str:
    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}
    tokens = [t for t in _norm_name(x).split() if t and t not in stop_tokens]
    return " ".join(sorted(tokens))

def _best_fuzzy_match(name_key: str, candidates: list[str]) -> tuple[str | None, float, str]:
    if not name_key or not candidates:
        return None, 0.0, "none"
    if HAS_RAPIDFUZZ:
        best_name, best_score, best_method = None, 0.0, "none"
        for scorer_name, scorer, threshold in [
            ("token_set", fuzz.token_set_ratio, FUZZY_TOKEN_SET_THRESHOLD),
            ("token_sort", fuzz.token_sort_ratio, FUZZY_TOKEN_SORT_THRESHOLD),
            ("partial", fuzz.partial_ratio, FUZZY_PARTIAL_THRESHOLD),
        ]:
            res = process.extractOne(name_key, candidates, scorer=scorer, score_cutoff=threshold)
            if res:
                cand, score, _ = res
                if float(score) > best_score:
                    best_name, best_score, best_method = cand, float(score), scorer_name
        if best_name is not None:
            return best_name, best_score, best_method
        res_any = process.extractOne(name_key, candidates, scorer=fuzz.token_set_ratio)
        if res_any:
            cand, score, _ = res_any
            return cand, float(score), "token_set_below_threshold"
        return None, 0.0, "none"

    best_name, best_score = None, 0.0
    for candidate in candidates:
        score = SequenceMatcher(None, name_key, candidate).ratio() * 100.0
        if score > best_score:
            best_name, best_score = candidate, score
    return best_name, best_score, "difflib"

def _build_extracted_df(existing_df: pd.DataFrame, new_rows_list: list, target_keys: set) -> pd.DataFrame:
    parts_local = []
    if not existing_df.empty:
        tmp_existing = existing_df.copy()
        tmp_existing["_target_name_key"] = tmp_existing["_name_key"]
        tmp_existing["name_match_type"] = tmp_existing.get("name_match_type", "partial_exact")
        tmp_existing["name_match_score"] = tmp_existing.get("name_match_score", 100.0)
        parts_local.append(tmp_existing)
    if new_rows_list:
        parts_local.append(pd.DataFrame(new_rows_list))

    if parts_local:
        out = pd.concat(parts_local, ignore_index=True)
        out["_name_key"] = out["_target_name_key"].fillna("")
        out = out[out["_name_key"].isin(target_keys)].copy()
        out = out.drop_duplicates(subset=["_name_key"], keep="first")
        return out
    return pd.DataFrame(columns=["_name_key"])

def _prepare_extracted_for_merge(extracted_df: pd.DataFrame) -> pd.DataFrame:
    # Remove helper/suffixed columns that can collide during merge.
    drop_cols = [c for c in extracted_df.columns if c in {"_name_norm", "_name_norm_wiki"} or c.endswith("_wiki")]
    return extracted_df.drop(columns=drop_cols, errors="ignore")

def _write_checkpoint(pol_base: pd.DataFrame, existing_df: pd.DataFrame, new_rows_list: list, target_keys: set, out_path: Path) -> int:
    extracted_tmp = _build_extracted_df(existing_df, new_rows_list, target_keys)
    extracted_tmp = _prepare_extracted_for_merge(extracted_tmp)
    final_tmp = pol_base.merge(extracted_tmp, on="_name_key", how="left", suffixes=("", "_wiki"))
    final_tmp = final_tmp.drop(columns=["_name_key", "_name_norm"], errors="ignore")
    final_tmp = final_tmp.drop(columns=[c for c in final_tmp.columns if str(c).endswith("_wiki")], errors="ignore")
    final_tmp.to_excel(out_path, index=False)
    return len(final_tmp)

source_cols = pd.read_csv(POLARISATION_SCORED_ONLY_PATH, nrows=0).columns.tolist()
pol_df = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()
pol_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
pol_name_col = next((c for c in pol_name_candidates if c in pol_df.columns), None)
if pol_name_col is None:
    raise ValueError(f"Could not find a name column in polarisation file. Columns: {list(pol_df.columns)}")

pol_df["_name_norm"] = pol_df[pol_name_col].fillna("").astype(str).map(_norm_name)
pol_df["_name_key"] = pol_df[pol_name_col].fillna("").astype(str).map(_match_key)
pol_df = pol_df[pol_df["_name_key"] != ""].copy()
pol_df = pol_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)
target_name_keys = set(pol_df["_name_key"].tolist())

reuse_candidates = [
    PARTIAL_PATH,
    FINAL_POLARISATION_PATH,
    FINAL_POLARISATION_MANUAL_PATH,
]

reuse_frames = []
reuse_paths_used = []
for candidate_path in reuse_candidates:
    if candidate_path.exists():
        try:
            tmp_df = pd.read_excel(candidate_path).copy()
            if not tmp_df.empty:
                reuse_frames.append(tmp_df)
                reuse_paths_used.append(str(candidate_path))
        except BaseException as e:
            print(f"Could not load reuse candidate {candidate_path} ({e}).")

if reuse_frames:
    partial_df = pd.concat(reuse_frames, ignore_index=True)
else:
    partial_df = pd.DataFrame()

if "name" in partial_df.columns:
    partial_name_col = "name"
elif "full_name_extracted" in partial_df.columns:
    partial_name_col = "full_name_extracted"
elif "mep_name_raw" in partial_df.columns:
    partial_name_col = "mep_name_raw"
elif "mep_name_clean" in partial_df.columns:
    partial_name_col = "mep_name_clean"
else:
    partial_name_col = None

if partial_name_col is not None and not partial_df.empty:
    partial_df["_name_norm"] = partial_df[partial_name_col].fillna("").astype(str).map(_norm_name)
    partial_df["_name_key"] = partial_df[partial_name_col].fillna("").astype(str).map(_match_key)
    existing_extracted_df = partial_df[partial_df["_name_key"].isin(target_name_keys)].copy()
    existing_extracted_df = existing_extracted_df.drop_duplicates(subset=["_name_key"], keep="first")
else:
    existing_extracted_df = pd.DataFrame()

processed_name_keys = set(existing_extracted_df.get("_name_key", pd.Series(dtype=str)).tolist())
missing_name_keys = [k for k in pol_df["_name_key"].tolist() if k not in processed_name_keys]

print("Reuse files considered:", len(reuse_paths_used))
if reuse_paths_used:
    for p in reuse_paths_used:
        print("  -", p)

print("Rapidfuzz available:", HAS_RAPIDFUZZ)
print("RUN_EXTRACTION:", RUN_EXTRACTION)
print("Polarisation-scored targets:", len(target_name_keys))
print("Already extracted from existing files:", len(processed_name_keys))
print("Need new extraction:", len(missing_name_keys))

source_df = pd.read_excel(INPUT_PATH).copy()
if "name" not in source_df.columns:
    raise ValueError("INPUT_PATH must contain a 'name' column for matching.")
source_df["_name_norm"] = source_df["name"].fillna("").astype(str).map(_norm_name)
source_df["_name_key"] = source_df["name"].fillna("").astype(str).map(_match_key)
source_df = source_df[source_df["_name_key"] != ""].copy()
source_df = source_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)

source_key_set = set(source_df["_name_key"].tolist())
source_key_list = source_df["_name_key"].tolist()

mapped_rows = []
exact_matches, fuzzy_matches, unmatched = 0, 0, 0

for tkey in missing_name_keys:
    if tkey in source_key_set:
        mapped_rows.append({"_target_name_key": tkey, "_source_name_key": tkey, "match_type": "exact", "match_score": 100.0})
        exact_matches += 1
    else:
        best_key, best_score, method = _best_fuzzy_match(tkey, source_key_list)
        if best_key is not None and method not in {"none", "token_set_below_threshold"}:
            mapped_rows.append({"_target_name_key": tkey, "_source_name_key": best_key, "match_type": f"fuzzy_{method}", "match_score": float(best_score)})
            fuzzy_matches += 1
        else:
            unmatched += 1
            if best_key is not None:
                mapped_rows.append({"_target_name_key": tkey, "_source_name_key": best_key, "match_type": "below_threshold", "match_score": float(best_score)})

map_df = pd.DataFrame(mapped_rows)
if not map_df.empty:
    map_df = map_df.drop_duplicates(subset=["_target_name_key"], keep="first")

print("Name mapping stats -> exact:", exact_matches, "fuzzy:", fuzzy_matches, "unmatched:", unmatched)

if not map_df.empty:
    low = map_df[map_df["match_type"].isin(["below_threshold"])]
    if not low.empty:
        print("Sample below-threshold matches (top 10):")
        display(low.sort_values("match_score", ascending=False).head(10))

if not map_df.empty:
    todo_df = map_df[map_df["match_type"] != "below_threshold"].merge(source_df, left_on="_source_name_key", right_on="_name_key", how="left")
else:
    todo_df = pd.DataFrame()

print("Matched source rows for missing names:", len(todo_df))

new_rows = []
interrupted = False
if RUN_EXTRACTION:
    total = len(todo_df)
    try:
        for i, (_, row) in enumerate(todo_df.iterrows(), start=1):
            name_hint = str(row.get("name", "") or "")
            target_key = str(row.get("_target_name_key", "") or "")
            print(f"[{i}/{total}] Working on: {name_hint} (target_key={target_key})")

            lang = str(row.get("wikipedia_language", "") or "")
            html_en = row.get("wikipedia_content_html_en")
            html_local = row.get("wikipedia_content_html")
            text_en = html_to_text(html_en) if isinstance(html_en, str) else ""
            text_local = html_to_text(html_local) if isinstance(html_local, str) else ""
            text = text_en + "\n\n" + text_local

            ts = datetime.now(timezone.utc).isoformat()
            extracted = None
            try:
                extracted = extract_with_llm(text, name_hint=name_hint, lang=lang)
                flat = flatten_extract(extracted)
                mep_flag = bool(extracted.get("mep_since") or extracted.get("ep_group")) or is_mep_from_text(text)
                quality = extraction_quality_flags(extracted)
                flat["MEP"] = bool(mep_flag)
                flat["extract_error"] = pd.NA
                flat["needs_manual_review"] = bool(len(quality["missing_required_fields"]) > 0)
                flat["missing_required_fields"] = json.dumps(quality["missing_required_fields"], ensure_ascii=False)
                err_msg = None
                status = "ok"
            except Exception as e:
                err_msg = str(e)
                status = "error"
                flat = {
                    "MEP": pd.NA,
                    "extract_error": err_msg,
                    "needs_manual_review": True,
                    "missing_required_fields": json.dumps(REQUIRED_EXTRACT_FIELDS, ensure_ascii=False),
                }

            flat["_target_name_key"] = target_key
            flat["_name_key"] = target_key
            flat["name_match_type"] = row.get("match_type", pd.NA)
            flat["name_match_score"] = row.get("match_score", pd.NA)
            merged_row = {**row.to_dict(), **flat}
            new_rows.append(merged_row)

            append_audit_jsonl({
                "timestamp_utc": ts,
                "name": name_hint,
                "target_name_key": target_key,
                "wikipedia_language": lang,
                "model_used": globals().get("LAST_MODEL_USED", MODEL),
                "prompt_version": PROMPT_VERSION,
                "error_status": status,
                "error_message": err_msg,
                "raw_response": None if extracted is None else json.dumps(extracted, ensure_ascii=False),
                "parsed_output": None if extracted is None else extracted,
                "missing_required_fields": [] if extracted is None else extraction_quality_flags(extracted)["missing_required_fields"],
                "needs_manual_review": bool(status != "ok" or (extracted is not None and len(extraction_quality_flags(extracted)["missing_required_fields"]) > 0)),
            })

            if CHECKPOINT_EVERY > 0 and (i % CHECKPOINT_EVERY == 0 or i == total):
                written_rows = _write_checkpoint(pol_base=pol_df, existing_df=existing_extracted_df, new_rows_list=new_rows, target_keys=target_name_keys, out_path=FINAL_POLARISATION_PATH)
                print(f"Checkpoint saved at {i}/{total}. Rows currently written: {written_rows}")
    except KeyboardInterrupt:
        interrupted = True
        print("Interrupted by user. Saving collected progress...")
        written_rows = _write_checkpoint(pol_base=pol_df, existing_df=existing_extracted_df, new_rows_list=new_rows, target_keys=target_name_keys, out_path=FINAL_POLARISATION_PATH)
        print(f"Checkpoint written after interrupt. Rows currently written: {written_rows}")
else:
    print("RUN_EXTRACTION=False -> mapping only, no LLM calls made.")

new_extracted_df = pd.DataFrame(new_rows)
extracted_df = _build_extracted_df(existing_extracted_df, new_rows, target_name_keys)
extracted_df = _prepare_extracted_for_merge(extracted_df)
final_df = pol_df.merge(extracted_df, on="_name_key", how="left", suffixes=("", "_wiki"))
final_df = final_df.drop(columns=["_name_key", "_name_norm"], errors="ignore")
final_df = final_df.drop(columns=[c for c in final_df.columns if str(c).endswith("_wiki")], errors="ignore")
allowed_extra_cols = [
    "name",
    "wikipedia_language",
    "page_title",
    "page_url",
    "wikipedia_content_html",
    "wikipedia_content_wikitext",
    "page_title_en",
    "page_url_en",
    "wikipedia_content_html_en",
    "wikipedia_content_wikitext_en",
    "Unnamed: 10",
    "Country",
    "full_name_extracted",
    "birth_date_extracted",
    "place_of_birth_extracted",
    "country_of_origin_extracted",
    "ep_group_extracted",
    "mep_since_extracted",
    "occupations_extracted",
    "parties_extracted",
    "education_extracted",
    "confidence_extracted",
    "gender_bucket",
    "region_bucket",
    "education_level_bucket",
    "education_field_bucket",
    "professional_background_bucket",
    "religion_bucket",
    "parental_status_bucket",
    "migration_background_bucket",
    "international_experience_bucket",
    "military_background_bucket",
    "prior_political_level_bucket",
    "demography_expertise_bucket",
    "evidence_birth_date",
    "evidence_country_of_origin",
    "evidence_ep_group",
    "evidence_education_level_bucket",
    "evidence_education_field_bucket",
    "evidence_professional_background_bucket",
    "evidence_parental_status_bucket",
    "evidence_migration_background_bucket",
    "evidence_international_experience_bucket",
    "evidence_prior_political_level_bucket",
    "evidence_demography_expertise_bucket",
    "evidence_education",
    "evidence_military_background_bucket",
    "MEP",
    "religion",
    "_target_name_key",
    "name_match_type",
    "name_match_score",
    "_source_name_key",
    "match_type",
    "match_score",
    "extract_error",
    "needs_manual_review",
    "missing_required_fields",
]
base_export_cols = [c for c in source_cols if c in final_df.columns]
extra_export_cols = [c for c in allowed_extra_cols if c in final_df.columns and c not in base_export_cols]
final_df = final_df.loc[:, base_export_cols + extra_export_cols]
final_df.to_excel(FINAL_POLARISATION_PATH, index=False)

assert len(final_df) > 0, "Final extracted output is empty"
assert FINAL_POLARISATION_PATH.exists(), f"Missing output: {FINAL_POLARISATION_PATH}"

if FINAL_POLARISATION_MANUAL_PATH.exists():
    print(f"Manual-improved file retained: {FINAL_POLARISATION_MANUAL_PATH}")

print("Newly extracted in this run:", len(new_extracted_df))
print("Final rows saved (should equal scored targets):", len(final_df))
if interrupted:
    print("Run ended by user interrupt after saving progress.")
print(f"Wrote: {FINAL_POLARISATION_PATH}")
print(f"Audit log: {LLM_AUDIT_JSONL_PATH}")

Reuse files considered: 1
  - C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\extracted_polarisation_final.xlsx
Rapidfuzz available: True
RUN_EXTRACTION: True
Polarisation-scored targets: 158
Already extracted from existing files: 155
Need new extraction: 3
Name mapping stats -> exact: 0 fuzzy: 3 unmatched: 0
Matched source rows for missing names: 3
[1/3] Working on: KARIŅŠ Krišjānis (target_key=arturs karins krisjanis)
Checkpoint saved at 1/3. Rows currently written: 158
[2/3] Working on: LÁSZLÓ András (target_key=andor laszlo)
Checkpoint saved at 2/3. Rows currently written: 158
[3/3] Working on: MARAN Pierfrancesco (target_key=majorino pierfrancesco)
Checkpoint saved at 3/3. Rows currently written: 158
Newly extracted in this run: 3
Final rows saved (should equal scored targets): 158
Wrote: C:\Users\Gebruiker\OneDrive - U

In [15]:

# ------------------------------------------------------------
# Snapshot current in-memory progress after interrupt
# ------------------------------------------------------------

required_vars = [
    "new_rows",
    "pol_df",
    "existing_extracted_df",
    "target_name_keys",
    "FINAL_POLARISATION_PATH",
    "_build_extracted_df",
    "_write_checkpoint",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    print("Missing variables in memory:", missing)
    print("Run Cell 7 first to initialize extraction state.")
else:
    print("In-memory new rows collected this run:", len(new_rows))
    print("Rows already reused from partial:", len(existing_extracted_df))

    # Force-save checkpoint from current in-memory state.
    written_rows = _write_checkpoint(
        pol_base=pol_df,
        existing_df=existing_extracted_df,
        new_rows_list=new_rows,
        target_keys=target_name_keys,
        out_path=FINAL_POLARISATION_PATH,
    )
    print("Checkpoint written from memory. Total rows in final file:", written_rows)
    print(f"Checkpoint path: {FINAL_POLARISATION_PATH}")

    # Build extracted view from memory and report coverage.
    extracted_now = _build_extracted_df(existing_extracted_df, new_rows, target_name_keys)
    print("Extracted rows currently available in memory:", len(extracted_now))

    cols = [c for c in ["name", "_target_name_key", "name_match_type", "name_match_score", "extract_error", "full_name_extracted"] if c in pd.DataFrame(new_rows).columns]
    if len(new_rows) > 0 and cols:
        print("Sample of in-memory rows (last 10):")
        display(pd.DataFrame(new_rows)[cols].tail(10))


Missing variables in memory: ['_build_extracted_df', '_write_checkpoint']
Run Cell 7 first to initialize extraction state.


In [60]:

# ------------------------------------------------------------
# Add old extracted-file MEP names to final polarisation file
# ------------------------------------------------------------

def _norm_name_local(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

final_df = pd.read_excel(FINAL_POLARISATION_PATH)
partial_df = pd.read_excel(PARTIAL_PATH) if PARTIAL_PATH.exists() else pd.DataFrame()

# Determine name columns in final and partial files.
final_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
final_name_col = next((c for c in final_name_candidates if c in final_df.columns), None)
if final_name_col is None:
    raise ValueError(f"No name column found in final file: {list(final_df.columns)}")

partial_name_candidates = ["name", "full_name_extracted", "mep_name_raw", "mep_name_clean"]
partial_name_col = next((c for c in partial_name_candidates if c in partial_df.columns), None)
if partial_name_col is None:
    print("No partial extracted file with an old-name column was found. Falling back to the current final names.")
    partial_df = final_df[[final_name_col]].copy()
    partial_df = partial_df.rename(columns={final_name_col: "mep_name_old_from_extracted"})
    partial_name_col = "mep_name_old_from_extracted"

final_df["_name_norm"] = final_df[final_name_col].fillna("").astype(str).map(_norm_name_local)
partial_df["_name_norm"] = partial_df[partial_name_col].fillna("").astype(str).map(_norm_name_local)

# Keep one old-name record per normalized name.
partial_names = (
    partial_df[["_name_norm", partial_name_col]]
    .dropna(subset=["_name_norm"])
    .drop_duplicates(subset=["_name_norm"], keep="first")
    .rename(columns={partial_name_col: "mep_name_old_from_extracted"})
 )
old_name_lookup = partial_names.set_index("_name_norm")["mep_name_old_from_extracted"] if not partial_names.empty else pd.Series(dtype=object)
if "mep_name_old_from_extracted" not in final_df.columns:
    final_df["mep_name_old_from_extracted"] = pd.NA
final_df["mep_name_old_from_extracted"] = final_df["mep_name_old_from_extracted"].fillna(final_df["_name_norm"].map(old_name_lookup))
final_df = final_df.drop(columns=["_name_norm"], errors="ignore")

final_df.to_excel(FINAL_POLARISATION_PATH, index=False)
print("Added old extracted names column: mep_name_old_from_extracted")
print(f"Wrote: {FINAL_POLARISATION_PATH}")


No partial extracted file with an old-name column was found. Falling back to the current final names.
Added old extracted names column: mep_name_old_from_extracted
Wrote: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\extracted_polarisation_final.xlsx


## Match Audit: Polarisation-Scored MEPs vs Extracted Wikipedia MEPs

This check uses the polarisation-scored MEP list as the base and verifies which of those have a matched row in `extracted_polarisation_final.xlsx`.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [43]:

# ------------------------------------------------------------
# Audit coverage from scored MEPs -> extracted final file
# ------------------------------------------------------------

def _norm_name_audit(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def _match_key_audit(x: str) -> str:
    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}
    tokens = [t for t in _norm_name_audit(x).split() if t and t not in stop_tokens]
    return " ".join(sorted(tokens))

scored_df = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()
scored_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
scored_name_col = next((c for c in scored_name_candidates if c in scored_df.columns), None)
if scored_name_col is None:
    raise ValueError(f"No name column found in scored file. Columns: {list(scored_df.columns)}")

scored_df["_name_key"] = scored_df[scored_name_col].fillna("").astype(str).map(_match_key_audit)
scored_df = scored_df[scored_df["_name_key"] != ""].copy()
scored_df = scored_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)

if not FINAL_POLARISATION_PATH.exists():
    raise FileNotFoundError(f"Missing file: {FINAL_POLARISATION_PATH}")

final_df = final_df.copy() if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and not final_df.empty else pd.read_excel(FINAL_POLARISATION_PATH).copy()
final_name_candidates = ["mep_name_raw", "mep_name_clean", "name", "full_name_extracted"]
final_name_col = next((c for c in final_name_candidates if c in final_df.columns), None)
if final_name_col is None:
    raise ValueError(f"No name column found in final file. Columns: {list(final_df.columns)}")

if "_name_key" in final_df.columns:
    final_df["_name_key"] = final_df["_name_key"].fillna("").astype(str)
else:
    final_df["_name_key"] = final_df[final_name_col].fillna("").astype(str).map(_match_key_audit)

extracted_signal_cols = [
    "full_name_extracted",
    "birth_date_extracted",
    "country_of_origin_extracted",
    "education_extracted",
    "ep_group_extracted",
    "confidence_extracted",
]
present_signal_cols = [c for c in extracted_signal_cols if c in final_df.columns]
if not present_signal_cols:
    raise ValueError(
        "Could not find extracted signal columns in final file. "
        f"Expected any of: {extracted_signal_cols}"
    )

final_df["has_wikipedia_match"] = final_df[present_signal_cols].notna().any(axis=1)
match_lookup = final_df.groupby("_name_key", dropna=False)["has_wikipedia_match"].max().to_dict()

audit_df = scored_df[[scored_name_col, "_name_key"]].copy()
audit_df["has_wikipedia_match"] = audit_df["_name_key"].map(match_lookup).fillna(False)

matched_count = int(audit_df["has_wikipedia_match"].sum())
total_count = int(len(audit_df))
unmatched_count = total_count - matched_count

print(f"Scored MEPs (base): {total_count}")
print(f"Scored MEPs with Wikipedia-extracted match: {matched_count}")
print(f"Scored MEPs without match: {unmatched_count}")

unmatched_df = audit_df[~audit_df["has_wikipedia_match"]].copy()
if unmatched_df.empty:
    print("All scored MEPs have a Wikipedia-extracted match.")
else:
    print("Sample unmatched scored MEPs (first 20):")
    display(unmatched_df.head(20))

audit_df.to_csv(AUDIT_OUTPUT_PATH, index=False)
print(f"Wrote audit file: {AUDIT_OUTPUT_PATH}")


Scored MEPs (base): 158
Scored MEPs with Wikipedia-extracted match: 155
Scored MEPs without match: 3
Sample unmatched scored MEPs (first 20):


,mep_name_raw,_name_key,has_wikipedia_match
17,Arturs Krišjānis Kariņš,arturs karins krisjanis,False
80,László Andor,andor laszlo,False
121,Pierfrancesco Majorino,majorino pierfrancesco,False


Wrote audit file: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\polarisation_vs_extracted_match_audit.csv


## Final manual adjustments

After the automated extraction workflow, I manually reviewed and corrected a small number of country irregularities.

The final hand-in file is:
- extracted_polarisation_final_manually_improved_1906.xlsx


